# Laboratorio 9 — Embeddings

**Milton Beltrán** · Natural Language Processing

**Corpus:** *Spanish News Classification* (Kaggle: [`kevinmorgado/spanish-news-classification`](https://www.kaggle.com/datasets/kevinmorgado/spanish-news-classification)) y *El ingenioso hidalgo don Quijote de la Mancha* (Kaggle: [`manuelmaaf97/quijote`](https://www.kaggle.com/datasets/manuelmaaf97/quijote)).
**Vectores preentrenados:** fastText SUC, tamaño M, 100 dimensiones ([dccuchile/spanish-word-embeddings](https://github.com/dccuchile/spanish-word-embeddings/blob/master/emb-from-suc.md)).

Volvemos al corpus de noticias de los laboratorios #1 a #4, pero tratado como en el #5, porque ahora lo que importa es el orden de las palabras. Word2vec aprende de las ventanas de vecinos, así que el texto se prepara como una lista de oraciones y no como una bolsa de palabras.
El Quijote se usa como segundo corpus para ver cómo cambian los vecinos de una misma palabra según el texto con el que se entrena.

| Labs #1–#4 (clasificación) | Lab #9 (embeddings) |
|---|---|
| Se eliminan las *stopwords* | Se conservan, porque son parte de la ventana de contexto |
| Stemming con `SnowballStemmer` | Ninguno, "gobiernos" y "gobernó" tienen que tener vectores distintos |
| Bolsa de palabras por documento | Lista de oraciones con los tokens en orden |
| Puntuación eliminada | También se elimina, si no ocuparía lugares de la ventana |

El archivo de fastText SUC pesa 1.1 GB y no se sube al repositorio. Hay que descargarlo de [Zenodo](https://zenodo.org/record/3234051/files/embeddings-m-model.vec?download=1) y dejarlo como `embeddings-m-model.vec` en la misma carpeta que este notebook.

## 0. Preparación del entorno y carga de los corpus

In [10]:
# --- 0.1 Librerías ---
# La variable de NLTK va antes del import: su hook de seguridad bloquea `regex` cuando
# el .venv vive dentro del directorio de trabajo.
import os

os.environ["NLTK_DISABLE_IMPORT_SECURITY"] = "1"

import re
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
import kagglehub

import gensim
from gensim.models import KeyedVectors
import sklearn
import torch

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("NLTK", nltk.__version__, "· gensim", gensim.__version__,
      "· scikit-learn", sklearn.__version__, "· PyTorch", torch.__version__)

NLTK 3.10.1 · gensim 4.4.0 · scikit-learn 1.7.2 · PyTorch 2.14.1


In [11]:
# --- 0.2 Recursos de NLTK ---
for recurso in ["punkt", "punkt_tab"]:
    nltk.download(recurso, quiet=True)

print("Recursos de NLTK listos.")

Recursos de NLTK listos.


In [12]:
# --- 0.3 Corpus de noticias ---
# Misma base de los labs #1–#4: 1,142 documentos tras quitar los duplicados exactos.
df = pd.read_csv("df_total.csv").drop_duplicates().reset_index(drop=True)

print(f"Documentos: {len(df):,} · Categorías: {df['Type'].nunique()}")
df.head(3)

Documentos: 1,142 · Categorías: 7


,url,news,Type
0,https://www.larepublica.co/redirect/post/3201905,Durante el foro La banca articulador empresari...,Otra
1,https://www.larepublica.co/redirect/post/3210288,El regulador de valores de China dijo el domin...,Regulaciones
2,https://www.larepublica.co/redirect/post/3240676,En una industria históricamente masculina como...,Alianzas


In [13]:
# --- 0.4 Quijote: descarga y recorte del aparato editorial ---
# Las mismas celdas 0.3–0.7 del Lab #5, condensadas.
ruta_txt = Path(kagglehub.dataset_download("manuelmaaf97/quijote")) / "don-quijote.txt"
crudo = ruta_txt.read_text(encoding="utf-8-sig")  # el archivo trae BOM

ini = crudo.index("*** START OF THIS PROJECT GUTENBERG EBOOK")
ini = crudo.index("\n", ini) + 1
fin = crudo.index("*** END OF THIS PROJECT GUTENBERG EBOOK")
libro = crudo[ini:fin].strip()
libro = libro[libro.index("El ingenioso hidalgo don Quijote de la Mancha"):
              libro.rindex("\nFin\n")].strip()

print(f"Cuerpo del libro: {len(libro):,} caracteres")

Cuerpo del libro: 2,097,945 caracteres


In [14]:
# --- 0.5 Quijote: párrafos y encabezados ---
# Wrap duro a ~76 columnas: hay que rearmar los párrafos antes de sentenizar.
# Los encabezados de capítulo se quitan; cada uno cuenta como un documento.
bloques = re.split(r"\n\s*\n", libro)
parrafos = [re.sub(r"\s+", " ", " ".join(b.split("\n")).strip()) for b in bloques]
parrafos = [p for p in parrafos if p]

ENCABEZADO = re.compile(r"^(Cap[íi]tulo\b|(Primera|Segunda|Tercera|Cuarta) parte del ingenioso)")
n_capitulos = sum(1 for p in parrafos if re.match(r"^Cap[íi]tulo\b", p))
parrafos_quijote = [p for p in parrafos if not ENCABEZADO.match(p)]

print(f"Capítulos: {n_capitulos} · Párrafos de prosa: {len(parrafos_quijote):,}")

Capítulos: 126 · Párrafos de prosa: 5,060


## 1. Preparación de los corpus

Pasamos del texto a una lista de oraciones, cada una con sus tokens en orden, que es el formato que espera gensim.

In [15]:
# --- 1.1 Normalización ---
# Minúsculas, sin puntuación, con tildes y números. Sin stopwords ni stemming (D-02).
LETRA = r"A-Za-zÁÉÍÓÚÜÑáéíóúüñ"
RAYA = re.compile(rf"(?<![{LETRA}])-|-(?![{LETRA}])")
APERTURA = re.compile(r"([¿¡«»])")
# "imaginamos.Para": párrafos pegados sin espacio en las noticias (P-01).
PEGADO = re.compile(r"([^\s][.!?])([A-ZÁÉÍÓÚÑ][a-záéíóúñ]|[¿¡])")
EEUU = re.compile(r"EE\s*\.\s*UU\b\.?")
MILES = re.compile(r"(?<=\d)\.(?=\d{3}\b)")  # SUC escribe "10000", no "10.000"
ALFANUM = re.compile(r"[a-záéíóúüñ0-9]")


def limpia(parrafo):
    parrafo = MILES.sub("", EEUU.sub("EEUU", parrafo))
    parrafo = PEGADO.sub(r"\1 \2", parrafo)
    return APERTURA.sub(r" \1 ", RAYA.sub(" - ", parrafo))


def tokeniza(oracion):
    """Minúsculas y solo tokens con alguna letra o dígito."""
    tokens = word_tokenize(oracion.lower(), language="spanish")
    return [t.strip(".…") for t in tokens if ALFANUM.search(t)]


demo = "Yo creo que el no tener un concepto de negocio sostenible puede tener un mayor impacto de lo que imaginamos.Para García el reto en EE.UU. es otro: 18.000 empleos."
print(sent_tokenize(demo, language="spanish"))
print([tokeniza(o) for o in sent_tokenize(limpia(demo), language="spanish")])

['Yo creo que el no tener un concepto de negocio sostenible puede tener un mayor impacto de lo que imaginamos.Para García el reto en EE.UU. es otro: 18.000 empleos.']
[['yo', 'creo', 'que', 'el', 'no', 'tener', 'un', 'concepto', 'de', 'negocio', 'sostenible', 'puede', 'tener', 'un', 'mayor', 'impacto', 'de', 'lo', 'que', 'imaginamos'], ['para', 'garcía', 'el', 'reto', 'en', 'eeuu', 'es', 'otro', '18000', 'empleos']]


In [16]:
# --- 1.2 Segmentación en oraciones y tokenización ---
def a_oraciones(textos):
    oraciones = []
    for texto in textos:
        for parrafo in texto.split("\n"):
            parrafo = limpia(parrafo.strip())
            if not parrafo:
                continue
            for o in sent_tokenize(parrafo, language="spanish"):
                tokens = tokeniza(o)
                if tokens:
                    oraciones.append(tokens)
    return oraciones


oraciones_noticias = a_oraciones(df["news"])
oraciones_quijote = a_oraciones(parrafos_quijote)

print(f"Noticias: {len(oraciones_noticias):,} oraciones")
print(f"Quijote:  {len(oraciones_quijote):,} oraciones")
print()
print(oraciones_noticias[5])
print(oraciones_quijote[100])

Noticias: 20,909 oraciones
Quijote:  10,585 oraciones

['el', 'directivo', 'argumentó', 'que', 'lo', 'que', 'se', 'encuentra', 'en', 'juego', 'acá', 'no', 'es', 'un', 'tema', 'de', 'rentabilidad', 'o', 'de', 'negocios', 'en', 'particular', 'es', 'un', 'tema', 'de', 'viabilidad', 'del', 'mundo', 'de', 'los', 'negocios', 'y', 'del', 'mundo', 'físico', 'en', 'general', 'como', 'lo', 'conocemos']
['soneto']


In [17]:
# --- 1.3 ¿SUC distingue mayúsculas y tildes? ---
# Se carga una sola vez; el resto del laboratorio reutiliza `suc`.
t0 = time.time()
suc = KeyedVectors.load_word2vec_format("embeddings-m-model.vec", limit=200_000)
print(f"SUC: {len(suc.key_to_index):,} palabras × {suc.vector_size} dims ({time.time() - t0:.0f} s)\n")

consultas = ["españa", "España", "espana", "gobierno", "Gobierno", "mañana", "manana",
             "eeuu", "2019", "10.000", "10000", ",", "¿"]
pd.DataFrame({
    "palabra": consultas,
    "en SUC": [w in suc.key_to_index for w in consultas],
    "rango": [suc.key_to_index.get(w, "—") for w in consultas],
})

SUC: 200,000 palabras × 100 dims (9 s)



,palabra,en SUC,rango
0,españa,True,541
1,España,False,—
2,espana,True,67448
3,gobierno,True,236
4,Gobierno,False,—
5,mañana,True,303
6,manana,True,31315
7,eeuu,True,6240
8,2019,True,7628
9,10.000,False,—


In [18]:
# --- 1.3 (cont.) Palabras con mayúscula en todo el vocabulario cargado ---
con_mayuscula = [w for w in suc.index_to_key if w != w.lower()]
print(f"Palabras con alguna mayúscula: {len(con_mayuscula)} de {len(suc.index_to_key):,}")
print("Las 10 más frecuentes de SUC:", suc.index_to_key[:10])

Palabras con alguna mayúscula: 0 de 200,000
Las 10 más frecuentes de SUC: ['</s>', '.', 'de', ',', 'la', 'que', 'el', 'en', 'y', 'a']


In [19]:
# --- 1.4 Tabla de los corpus ---
def resumen(nombre, documentos, oraciones):
    conteo = Counter(t for o in oraciones for t in o)
    n_tokens = sum(conteo.values())
    return {"corpus": nombre, "documentos": documentos, "oraciones": len(oraciones),
            "tokens": n_tokens, "tipos": len(conteo),
            "tokens por oración": round(n_tokens / len(oraciones), 1)}


tabla_corpus = pd.DataFrame([
    resumen("Noticias", len(df), oraciones_noticias),
    resumen("Quijote (capítulos)", n_capitulos, oraciones_quijote),
]).set_index("corpus")
tabla_corpus

,documentos,oraciones,tokens,tipos,tokens por oración
corpus,,,,,
Noticias,1142,20909,605523,30750,29.0
Quijote (capítulos),126,10585,378666,22842,35.8


In [20]:
# --- 1.5 Vocabulario según min_count ---
filas = []
for nombre, oraciones in [("Noticias", oraciones_noticias), ("Quijote", oraciones_quijote)]:
    conteo = Counter(t for o in oraciones for t in o)
    n_tokens = sum(conteo.values())
    for k in (1, 2, 5):
        fuera = sum(f for f in conteo.values() if f < k)
        filas.append({"corpus": nombre, "min_count": k,
                      "vocabulario": sum(1 for f in conteo.values() if f >= k),
                      "% tokens fuera": round(100 * fuera / n_tokens, 2)})

pd.DataFrame(filas).set_index(["corpus", "min_count"])

vocabulario  % tokens fuera
corpus   min_count                             
Noticias 1                30750            0.00
         2                16832            2.30
         5                 8595            5.91
Quijote  1                22842            0.00
         2                11690            2.95
         5                 5275            7.40

In [21]:
# --- 1.6 Pares skip-gram a mano, ventana 2 ---
def pares_skipgram(tokens, ventana):
    pares = []
    for i, central in enumerate(tokens):
        for j in range(max(0, i - ventana), min(len(tokens), i + ventana + 1)):
            if j != i:
                pares.append((central, tokens[j]))
    return pares


oracion = next(o for o in oraciones_noticias if " ".join(o) == "nunca pensé que pudiera trabajar en un banco")
pares = pares_skipgram(oracion, ventana=2)

print(" ".join(oracion), f"→ {len(oracion)} tokens, {len(pares)} pares\n")
pd.DataFrame(pares, columns=["central", "contexto"]).groupby("central", sort=False)["contexto"].apply(list)

nunca pensé que pudiera trabajar en un banco → 8 tokens, 26 pares



central
nunca                            [pensé, que]
pensé                   [nunca, que, pudiera]
que         [nunca, pensé, pudiera, trabajar]
pudiera            [pensé, que, trabajar, en]
trabajar               [que, pudiera, en, un]
en             [pudiera, trabajar, un, banco]
un                      [trabajar, en, banco]
banco                                [en, un]
Name: contexto, dtype: object

In [22]:
# --- 1.7 Pares de entrenamiento de todo el corpus de noticias ---
n_tokens = sum(len(o) for o in oraciones_noticias)
filas = []
for ventana in (2, 5):
    n_pares = sum(len(pares_skipgram(o, ventana)) for o in oraciones_noticias)
    filas.append({"ventana": ventana, "pares": n_pares, "tokens": n_tokens,
                  "pares por token": round(n_pares / n_tokens, 2),
                  "máximo teórico (2·ventana)": 2 * ventana})

pd.DataFrame(filas).set_index("ventana")

,pares,tokens,pares por token,máximo teórico (2·ventana)
ventana,,,,
2,2296962,605523,3.79,4
5,5433568,605523,8.97,10


### 1.8 Resultados y discusión

#### ¿Vale la pena quitar stopwords o lematizar antes de entrenar embeddings?

No. En los labs #1 a #4 las quitábamos porque el clasificador solo veía conteos por documento, y palabras como "de" o "que" no decían nada de la categoría. Acá la unidad es la ventana, entonces si se borran las stopwords, palabras que estaban a cinco posiciones de distancia quedan juntas y el modelo aprende vecinos que en el texto nunca lo fueron. Además las stopwords cargan sintaxis, por ejemplo el "la" antes de un sustantivo es lo que le dice al modelo que la palabra es femenina.

Con el stemming pasa algo parecido. "gobiernos" y "gobernó" terminan los dos en `gobern` y se pierden el número y el tiempo verbal, que son justo dos de los tipos de analogía que se evalúan en la sección 5.

La normalización quedó en minúsculas, sin puntuación, con tildes y con números. La puntuación se quita al revés que en el Lab #5, porque allá era algo que el modelo tenía que predecir y acá una coma solo ocupa un lugar de la ventana sin aportar contexto.
Las noticias necesitaron tres arreglos más, los párrafos pegados sin espacio como "imaginamos.Para" (6,147 casos en 844 documentos) que Punkt no separa, "EE.UU." que partía la oración en dos y el separador de miles.

#### ¿Distingue SUC mayúsculas y tildes?

Mayúsculas no. De las 200,000 palabras que cargamos ninguna tiene mayúscula, así que "España" no existe en el modelo y pasar todo a minúsculas es obligatorio. Las tildes sí las distingue, "españa" y "espana" son dos entradas distintas y la segunda es mucho más rara (posición 541 contra 67,448), seguramente de texto escrito sin tildes. Si las quitáramos, las consultas irían a parar a esos vectores ruidosos.

SUC tampoco usa separador de miles, tiene "10000" pero no "10.000", por eso también se quita el punto.

#### Tamaño de los corpus

| Corpus | Documentos | Oraciones | Tokens | Tipos |
|---|---|---|---|---|
| Noticias | 1,142 | 20,909 | 605,523 | 30,750 |
| Quijote | 126 capítulos | 10,585 | 378,666 | 22,842 |

| min_count | Vocab. noticias | % tokens fuera | Vocab. Quijote | % tokens fuera |
|---|---|---|---|---|
| 1 | 30,750 | 0.00 | 22,842 | 0.00 |
| 2 | 16,832 | 2.30 | 11,690 | 2.95 |
| 5 | 8,595 | 5.91 | 5,275 | 7.40 |

Con `min_count = 2` el vocabulario baja casi a la mitad y solo se pierde el 2.3% de los tokens. Esto es porque el 45% de las palabras de las noticias aparece una sola vez, y con una sola aparición no se puede aprender un vector confiable.

#### ¿Por qué word2vec no necesita datos etiquetados?

La oración de 8 tokens de la celda 1.6 da 26 pares, y el corpus completo da **2,296,962** con ventana 2 y **5,433,568** con ventana 5. Eso es 3.8 y 9 pares por token, casi el máximo de 4 y 10, y lo que falta son los bordes de cada oración donde no hay vecinos de un lado.

Cada par funciona como un ejemplo de entrenamiento donde la "etiqueta" es la palabra de contexto, y esa etiqueta ya viene en el mismo texto. Por eso se dice que es auto-supervisado, cualquier texto sirve y nadie tiene que anotar nada.
Así fue como SUC se pudo entrenar con 2,600 millones de palabras.

## 2. Co-ocurrencia, PPMI y SVD

Antes de entrenar cualquier red, aplicamos la hipótesis distribucional de la forma más directa, contando qué palabras aparecen cerca de cuáles. Esta sección usa solo el corpus de noticias.

In [23]:
# --- 2.1 Matriz de co-ocurrencia ±2 sobre las 2,000 palabras más frecuentes ---
from numpy.linalg import norm

conteo_noticias = Counter(t for o in oraciones_noticias for t in o)
vocab_cooc = [w for w, _ in conteo_noticias.most_common(2000)]
idx_cooc = {w: i for i, w in enumerate(vocab_cooc)}

X_cooc = np.zeros((len(vocab_cooc), len(vocab_cooc)), dtype=np.float64)
for o in oraciones_noticias:
    for central, contexto in pares_skipgram(o, ventana=2):
        if central in idx_cooc and contexto in idx_cooc:
            X_cooc[idx_cooc[central], idx_cooc[contexto]] += 1

ceros = (X_cooc == 0).mean()
print(f"Forma: {X_cooc.shape} · ceros: {ceros:.2%} · simétrica: {np.allclose(X_cooc, X_cooc.T)}")
print("BoW del Lab #2 (documentos × vocabulario completo): 98.71 % ceros")

Forma: (2000, 2000) · ceros: 93.78% · simétrica: True
BoW del Lab #2 (documentos × vocabulario completo): 98.71 % ceros


In [24]:
# --- 2.2 Coseno entre filas de conteos ---
def coseno(u, v):
    return float(u @ v / (norm(u) * norm(v)))


def fila(M, w):
    return M[idx_cooc[w]]


pares_prueba = [("empresa", "compañía"), ("gobierno", "ministerio"), ("banco", "financiera"),
                ("empresa", "año"), ("tecnología", "presidente")]

pd.DataFrame([{"par": f"{a} / {b}", "coseno conteos": round(coseno(fila(X_cooc, a), fila(X_cooc, b)), 3)}
              for a, b in pares_prueba]).set_index("par")

,coseno conteos
par,
empresa / compañía,0.962
gobierno / ministerio,0.783
banco / financiera,0.362
empresa / año,0.445
tecnología / presidente,0.671


In [25]:
# --- 2.3 Las 10 palabras con más co-ocurrencias ---
def top_fila(M, w, k=10):
    v = fila(M, w)
    orden = np.argsort(-v)
    return [f"{vocab_cooc[j]} ({v[j]:.0f})" if M is X_cooc else f"{vocab_cooc[j]} ({v[j]:.2f})"
            for j in orden if vocab_cooc[j] != w][:k]


pd.DataFrame({w: top_fila(X_cooc, w) for w in ["empresa", "gobierno"]})

,empresa,gobierno
0,la (192),el (214)
1,de (142),del (106)
2,una (62),de (83)
3,que (47),que (62)
4,en (42),y (56)
5,y (32),nacional (36)
6,con (27),en (29)
7,a (25),para (24)
8,para (17),al (24)
9,como (16),por (23)


In [26]:
# --- 2.4 PPMI con NumPy ---
# PPMI(w, c) = max(0, log2[P(w, c) / (P(w) · P(c))]); los conteos cero dan log(0) = -inf → 0.
def ppmi(X):
    total = X.sum()
    p_wc = X / total
    p_w = X.sum(axis=1, keepdims=True) / total
    p_c = X.sum(axis=0, keepdims=True) / total
    with np.errstate(divide="ignore"):
        pmi = np.log2(p_wc / (p_w * p_c))
    return np.maximum(pmi, 0)


X_ppmi = ppmi(X_cooc)
print(f"PPMI: {X_ppmi.shape} · ceros: {(X_ppmi == 0).mean():.2%} · máximo: {X_ppmi.max():.2f}")

PPMI: (2000, 2000) · ceros: 94.98% · máximo: 12.47


In [27]:
# --- 2.5 Las 10 palabras más asociadas, ahora con PPMI ---
pd.DataFrame({w: top_fila(X_ppmi, w) for w in ["empresa", "gobierno"]})

,empresa,gobierno
0,telecomunicaciones (5.51),corporativo (7.23)
1,llega (5.22),federal (5.57)
2,especializada (5.12),sánchez (5.39)
3,líder (4.96),nacional (5.26)
4,cualquier (4.71),iván (5.24)
5,estadounidense (4.70),funcionarios (5.17)
6,colombiana (4.59),petro (4.76)
7,criterios (4.59),estará (4.55)
8,española (4.50),llamado (4.52)
9,somos (4.28),duque (4.40)


In [28]:
# --- 2.5 (cont.) Los mismos pares de 2.2 con PPMI ---
pd.DataFrame([{"par": f"{a} / {b}",
               "conteos": round(coseno(fila(X_cooc, a), fila(X_cooc, b)), 3),
               "PPMI": round(coseno(fila(X_ppmi, a), fila(X_ppmi, b)), 3)}
              for a, b in pares_prueba]).set_index("par")

,conteos,PPMI
par,,
empresa / compañía,0.962,0.264
gobierno / ministerio,0.783,0.098
banco / financiera,0.362,0.131
empresa / año,0.445,0.050
tecnología / presidente,0.671,0.066


In [29]:
# --- 2.6 SVD a 100 dimensiones: primer modelo de embeddings ---
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=100, random_state=RANDOM_STATE)
vectores_svd = svd.fit_transform(X_ppmi)

kv_svd = KeyedVectors(vector_size=100)
kv_svd.add_vectors(vocab_cooc, vectores_svd)

print(f"Varianza explicada por 100 componentes: {svd.explained_variance_ratio_.sum():.1%}")
for w in ["empresa", "gobierno", "banco"]:
    print(f"{w:>9} →", [v for v, _ in kv_svd.most_similar(w, topn=5)])

Varianza explicada por 100 componentes: 25.0%
  empresa → ['compañía', 'plataforma', 'entidad', 'tecnología', 'asociación']
 gobierno → ['congreso', 'banco', 'república', 'central', 'también']
    banco → ['grupo', 'bbva', 'central', 'colombia', 'gobierno']


### 2.7 Resultados y discusión

La matriz de co-ocurrencia de las 2,000 palabras más frecuentes es de 2,000 × 2,000 y tiene **93.78%** de ceros. Es menos dispersa que la BoW del Lab #2 (98.71%), pero no porque contar vecinos dé matrices más densas, sino porque aquí solo dejamos las palabras frecuentes. Con las 30,750 palabras del vocabulario completo estaría mucho más vacía.

#### ¿Coincide el coseno con su intuición?

| Par | Conteos | PPMI |
|---|---|---|
| empresa / compañía | 0.962 | 0.264 |
| gobierno / ministerio | 0.783 | 0.098 |
| banco / financiera | 0.362 | 0.131 |
| empresa / año | 0.445 | 0.050 |
| tecnología / presidente | 0.671 | 0.066 |

Con los conteos solo a medias. "empresa / compañía" sale alto como esperaba, pero "tecnología / presidente" (0.671) queda muy por encima de "banco / financiera" (0.362), que deberían estar más relacionadas. Casi cualquier par de sustantivos se parece porque todos comparten los mismos vecinos, que son "el", "la" y "de".

#### ¿Qué tipo de palabras dominan la lista?

Las stopwords. Los 10 vecinos más frecuentes de "empresa" son la, de, una, que, en, y, con, a, para y como, y los de "gobierno" son casi los mismos con un "nacional" en medio. Eso no dice nada de lo que significa "empresa", lo único que dice es que es un sustantivo femenino.

#### ¿Qué cambió con PPMI y por qué?

PPMI compara cuántas veces aparecen juntas dos palabras contra cuántas veces se esperaría por puro azar, según la frecuencia de cada una. Como "la" aparece junto a todo, su PMI con "empresa" es bajo y desaparece de la lista.
Lo que queda son asociaciones más específicas, como "telecomunicaciones", "especializada", "líder" y "colombiana" para empresa, y "federal", "funcionarios", "sánchez", "petro" y "duque" para gobierno, que son justamente los gobiernos de los que hablan las noticias.

Los cosenos bajan todos, pero el orden se parece más a lo que uno esperaría. "empresa / compañía" sigue primero y los dos pares sin relación quedan al fondo. La lista igual trae algo de ruido ("llega", "cualquier"), ya que PMI le da demasiado peso a los pares raros y con pocos conteos cualquier coincidencia parece significativa.

#### ¿Por qué las filas ya son vectores de palabras y qué resuelven PPMI y SVD?

Porque cada fila describe a una palabra según los contextos donde aparece, y si dos palabras se usan en contextos parecidos sus filas también se parecen. Esa es la hipótesis distribucional.

El problema es que las filas tienen 2,000 dimensiones, casi todas en cero, y las dominan las palabras más frecuentes. PPMI resuelve lo segundo porque le quita el peso a las stopwords, y SVD resuelve lo primero comprimiendo las 2,000 columnas en 100 dimensiones densas que juntan contextos parecidos en una misma dirección. Así "empresa" y "compañía" pueden parecerse aunque no compartan exactamente los mismos vecinos, y de hecho los 5 vecinos de "empresa" con SVD ya son compañía, plataforma, entidad, tecnología y asociación.

## 3. Investigación

#### ¿En qué se diferencian CBOW y skip-gram? ¿Cuál se espera que funcione mejor con palabras poco frecuentes?

CBOW promedia los vectores de las palabras del contexto y con eso predice la palabra central, mientras que skip-gram hace lo contrario y usa la palabra central para predecir cada palabra del contexto. CBOW es más rápido, pero al promediar se diluyen las palabras raras.
En skip-gram cada vez que aparece una palabra rara se generan sus propios pares de entrenamiento, por lo tanto funciona mejor con palabras poco frecuentes.

#### ¿Qué problema resuelve negative sampling frente a un softmax sobre todo el vocabulario?

Con softmax, cada predicción tiene que normalizar sobre todo el vocabulario (16,832 palabras en nuestro caso y millones en SUC), y eso se paga en cada uno de los pares. Negative sampling cambia ese problema por varias clasificaciones binarias, donde se compara el par real contra *k* palabras sorteadas al azar.

P(real | w, c) = σ(v<sub>w</sub> · c<sub>c</sub>) = 1 / (1 + e<sup>−v<sub>w</sub> · c<sub>c</sub></sup>)

Aquí *v<sub>w</sub>* es el vector de la palabra central en la tabla de palabras y *c<sub>c</sub>* es el vector de la palabra de contexto en la tabla de contextos. Si el producto punto es alto, la probabilidad se acerca a 1 y el modelo cree que el par sí aparece en el texto.

#### ¿Por qué word2vec aprende dos tablas de vectores y cuál se conserva al final?

Porque una palabra casi nunca es su propio contexto. Con una sola tabla, el producto de una palabra consigo misma siempre sería alto y el modelo creería que cada palabra aparece al lado de sí misma, cosa que casi nunca pasa. Al final se conserva la tabla de palabras, que en gensim es `model.wv`.

#### ¿En qué se diferencian word2vec, GloVe y fastText? ¿Por qué fastText es especialmente útil en español?

Word2vec es predictivo y local, ya que aprende ventana por ventana. GloVe en cambio parte de la matriz global de co-ocurrencias, parecido al SVD de la sección 2, pero la factoriza con un objetivo de regresión ponderada.
FastText representa cada palabra como la suma de sus n-gramas de caracteres, y por eso puede armar un vector para palabras que nunca vio. En español eso ayuda mucho porque un solo verbo tiene decenas de conjugaciones y muchas de ellas no aparecen en el corpus.

#### Parámetros de `gensim.models.Word2Vec`

| Parámetro | Propósito |
|---|---|
| `vector_size` | Dimensión de los vectores |
| `window` | Cuántas palabras de contexto se toman a cada lado de la central |
| `min_count` | Frecuencia mínima para entrar al vocabulario |
| `sg` | 1 = skip-gram, 0 = CBOW |
| `negative` | Cuántas palabras negativas se sortean por cada par |
| `sample` | Umbral para descartar al azar las palabras muy frecuentes |
| `epochs` | Cuántas pasadas se hacen sobre el corpus |

#### `nn.Embedding`, `from_pretrained` y `nn.EmbeddingBag`

`nn.Embedding(num_embeddings, embedding_dim, padding_idx)` es una tabla entrenable con `num_embeddings` filas y `embedding_dim` columnas, y la fila `padding_idx` se queda en cero y no recibe gradiente. `nn.Embedding.from_pretrained(E, freeze=True)` la inicializa con vectores que ya fueron entrenados, y con `freeze=False` esos vectores también se ajustan durante el entrenamiento (fine-tuning).
`nn.EmbeddingBag` busca los vectores de una secuencia y los combina en uno solo en el mismo paso según `mode` (`"mean"`, `"sum"` o `"max"`), sin tener que armar la matriz intermedia.

Multiplicar un vector one-hot por la matriz E equivale a seleccionar una fila porque el one-hot tiene un único 1 y todo lo demás es cero. El producto solo deja la fila de E que está en esa posición, por eso `nn.Embedding` indexa directamente en vez de hacer la multiplicación.